# Text Retrieval Fine-Tuning: Match a Large Model with a 5x Smaller One

**Goal:** Show that a small, fine-tuned model can match or approach the performance of a much larger general-purpose model on domain-specific retrieval.

**Setup:**
- **Large model:** `BAAI/bge-base-en-v1.5` (110M params) — strong general-purpose retriever
- **Small model:** `sentence-transformers/all-MiniLM-L6-v2` (22M params, 5x smaller)
- **Dataset:** SciFact — scientific claim verification (809 train queries, 300 test queries)

**Plan:**
1. Evaluate both models as baselines
2. Fine-tune MiniLM three ways: random negatives (YAML/CLI), hard negatives (Python API), mixed negatives (Python API)
3. Compare all results — does fine-tuned MiniLM catch up to BGE?

In [ ]:
!pip install khoji

## Step 1: Baseline Evaluation

Let's see how both models perform out-of-the-box on SciFact — no fine-tuning, no tricks.

In [ ]:
import khoji

# Load SciFact test set
test_dataset = khoji.load_beir("scifact", split="test")
print(f"Test: {len(test_dataset.queries)} queries, {len(test_dataset.corpus)} docs")

In [ ]:
# Baseline 1: BGE-base (110M params) — the "big" model
bge_evaluator = khoji.Evaluator("BAAI/bge-base-en-v1.5")
bge_baseline = bge_evaluator.evaluate(
    dataset_name="scifact",
    k_values=[1, 5, 10],
    dataset=test_dataset,
)
bge_baseline.print()
del bge_evaluator

In [ ]:
# Baseline 2: MiniLM (22M params, 5x smaller) — the model we'll fine-tune
minilm_evaluator = khoji.Evaluator("sentence-transformers/all-MiniLM-L6-v2")
minilm_baseline = minilm_evaluator.evaluate(
    dataset_name="scifact",
    k_values=[1, 5, 10],
    dataset=test_dataset,
)
minilm_baseline.print()
del minilm_evaluator

In [ ]:
# Compare baselines
print("Baseline comparison (no fine-tuning):")
print(f"{'Metric':<12} {'BGE (110M)':>12} {'MiniLM (22M)':>14} {'Gap':>10}")
print("-" * 50)
for metric in bge_baseline.metrics:
    b = bge_baseline.metrics[metric]
    m = minilm_baseline.metrics[metric]
    print(f"{metric:<12} {b:>12.4f} {m:>14.4f} {m - b:>+10.4f}")

## Step 2: Fine-tune MiniLM — Three Negative Strategies

We'll fine-tune MiniLM three different ways to show the impact of negative mining strategy. Each approach also demonstrates a different way to use the library.

### Approach 1: Random Negatives via YAML Config

The simplest approach — write a YAML config and run it. No model needed for mining, fast setup.

In [ ]:
# Write a YAML config for random negatives
config_yaml = """
model:
  name: sentence-transformers/all-MiniLM-L6-v2

data:
  dataset: scifact
  split: train
  negatives: random
  n_negatives: 3

lora:
  r: 16
  alpha: 32
  dropout: 0.1

train:
  epochs: 5
  batch_size: 16
  grad_accum_steps: 1
  lr: 2e-5
  warmup_steps: 50
  max_length: 512
  loss: infonce
  temperature: 0.05
  sanity_check_samples: 5

seed: 42

eval:
  k_values: [1, 5, 10]
  split: test
  run_before: false
  run_after: true

output_dir: ./output/scifact-random
"""

with open("scifact_random.yaml", "w") as f:
    f.write(config_yaml)

print("Config written. Running training...")

In [ ]:
# Run via the Python API (equivalent to: khoji scifact_random.yaml)
config = khoji.ForgeConfig.from_yaml("scifact_random.yaml")
result_random = khoji.run(config)

print(f"\nFinal epoch loss: {result_random.history.epoch_loss[-1]:.4f}")

### Approach 2: Hard Negatives via Python API

Hard negatives are mined using the model's own embeddings — the most confusing non-relevant documents become training signal. This requires encoding the corpus first, but produces better negatives.

In [ ]:
from functools import partial
from khoji.loss import infonce_loss

# Load training data
train_dataset = khoji.load_beir("scifact", split="train")
print(f"Train: {len(train_dataset.queries)} queries, {len(train_dataset.corpus)} docs")

# Mine hard negatives using MiniLM's own embeddings
mining_model = khoji.EmbeddingModel("sentence-transformers/all-MiniLM-L6-v2")
hard_triplets = khoji.mine_hard_negatives(
    train_dataset, mining_model,
    n_negatives=3, top_k=50,
)
del mining_model
print(f"Mined {len(hard_triplets)} hard negative triplets")

In [ ]:
# Train with hard negatives
config_hard = khoji.TrainingConfig(
    epochs=5,
    batch_size=16,
    lr=2e-5,
    warmup_steps=50,
    max_length=512,
    loss_fn=partial(infonce_loss, temperature=0.05),
    lora=khoji.LoRASettings(r=16, alpha=32, dropout=0.1),
    save_dir="./output/scifact-hard/adapter",
    sanity_check_samples=5,
)

trainer_hard = khoji.Trainer("sentence-transformers/all-MiniLM-L6-v2", config_hard)
history_hard = trainer_hard.train(khoji.TripletDataset(hard_triplets))

print(f"\nFinal epoch loss: {history_hard.epoch_loss[-1]:.4f}")

In [ ]:
# Evaluate hard negatives model
eval_hard = khoji.Evaluator(
    "sentence-transformers/all-MiniLM-L6-v2",
    adapter_path="./output/scifact-hard/adapter",
)
result_hard_eval = eval_hard.evaluate(
    dataset_name="scifact", k_values=[1, 5, 10], dataset=test_dataset,
)
result_hard_eval.print()
del eval_hard

### Approach 3: Mixed Negatives via Python API

Mixed negatives combine the best of both worlds — easy random negatives for basic discrimination plus hard negatives for fine-grained ranking. This is the recommended approach.

In [ ]:
# Build mixed negatives: 2 random + 1 hard per (query, positive) pair
mining_model = khoji.EmbeddingModel("sentence-transformers/all-MiniLM-L6-v2")
mixed_triplets = khoji.build_mixed_negatives(
    train_dataset, mining_model,
    n_random=2, n_hard=1, top_k=50,
)
del mining_model
print(f"Built {len(mixed_triplets)} mixed triplets")

In [ ]:
# Train with mixed negatives
config_mixed = khoji.TrainingConfig(
    epochs=5,
    batch_size=16,
    lr=2e-5,
    warmup_steps=50,
    max_length=512,
    loss_fn=partial(infonce_loss, temperature=0.05),
    lora=khoji.LoRASettings(r=16, alpha=32, dropout=0.1),
    save_dir="./output/scifact-mixed/adapter",
    sanity_check_samples=5,
)

trainer_mixed = khoji.Trainer("sentence-transformers/all-MiniLM-L6-v2", config_mixed)
history_mixed = trainer_mixed.train(khoji.TripletDataset(mixed_triplets))

print(f"\nFinal epoch loss: {history_mixed.epoch_loss[-1]:.4f}")

In [ ]:
# Evaluate mixed negatives model
eval_mixed = khoji.Evaluator(
    "sentence-transformers/all-MiniLM-L6-v2",
    adapter_path="./output/scifact-mixed/adapter",
)
result_mixed_eval = eval_mixed.evaluate(
    dataset_name="scifact", k_values=[1, 5, 10], dataset=test_dataset,
)
result_mixed_eval.print()
del eval_mixed

## Step 3: Compare All Results

In [ ]:
results = {
    "BGE-base (110M)": bge_baseline.metrics,
    "MiniLM baseline": minilm_baseline.metrics,
    "MiniLM + random": result_random.finetuned.metrics,
    "MiniLM + hard": result_hard_eval.metrics,
    "MiniLM + mixed": result_mixed_eval.metrics,
}

# Print comparison table
header = f"{'Metric':<12}" + "".join(f"{name:>18}" for name in results)
print(header)
print("-" * len(header))

for metric in ["ndcg@10", "mrr@10", "recall@10"]:
    row = f"{metric:<12}"
    for name, metrics in results.items():
        row += f"{metrics[metric]:>18.4f}"
    print(row)

# Highlight the gap closure
bge_ndcg = bge_baseline.metrics["ndcg@10"]
minilm_ndcg = minilm_baseline.metrics["ndcg@10"]
best_ft_ndcg = max(
    result_random.finetuned.metrics["ndcg@10"],
    result_hard_eval.metrics["ndcg@10"],
    result_mixed_eval.metrics["ndcg@10"],
)
gap_before = bge_ndcg - minilm_ndcg
gap_after = bge_ndcg - best_ft_ndcg
print(f"\nnDCG@10 gap vs BGE: {gap_before:.4f} (before) → {gap_after:.4f} (after fine-tuning)")
if gap_before > 0:
    print(f"Gap closed by {100 * (1 - gap_after / gap_before):.1f}%")

## Step 4: Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

histories = {
    "Random": result_random.history,
    "Hard": history_hard,
    "Mixed": history_mixed,
}

for name, hist in histories.items():
    axes[0].plot(hist.step_loss, label=name, alpha=0.7)
    axes[1].plot(hist.step_lr, label=name, alpha=0.7)
    axes[2].plot(hist.step_grad_norm, label=name, alpha=0.7)

axes[0].set_title("Loss per Step")
axes[0].set_xlabel("Optimizer Step")
axes[0].legend()

axes[1].set_title("Learning Rate Schedule")
axes[1].set_xlabel("Optimizer Step")
axes[1].legend()

axes[2].set_title("Gradient Norm")
axes[2].set_xlabel("Optimizer Step")
axes[2].legend()

plt.tight_layout()
plt.savefig("scifact_training_curves.png", dpi=150)
plt.show()

## Step 5: Use the Fine-Tuned Model for Inference

In [ ]:
import torch

# Load the best fine-tuned model
model = khoji.EmbeddingModel(
    "sentence-transformers/all-MiniLM-L6-v2",
    adapter_path="./output/scifact-mixed/adapter",
)

# Encode a scientific claim
query = "Vitamin D deficiency is associated with increased risk of respiratory infections."
query_emb = model.encode([query])

# Encode some candidate documents
docs = [
    "Low serum vitamin D levels have been linked to susceptibility to acute respiratory tract infections in multiple observational studies.",
    "Calcium supplementation has no significant effect on bone density in adults over 50.",
    "Vitamin D supplementation reduced the risk of respiratory infection by 12% in a meta-analysis of 25 randomized controlled trials.",
    "Iron deficiency anemia is the most common nutritional disorder worldwide.",
]
doc_embs = model.encode(docs)

# Rank by similarity
scores = torch.mm(query_emb, doc_embs.t()).squeeze(0)
ranked = torch.argsort(scores, descending=True).tolist()

print(f"Query: {query}\n")
for rank, idx in enumerate(ranked):
    print(f"  Rank {rank + 1} (score={scores[idx]:.4f}): {docs[idx][:80]}...")

## Step 6: Iterative Mining Rounds (Advanced)

What if one round of hard negative mining isn't enough? With **mining rounds**, the pipeline automatically re-mines negatives using the fine-tuned model from the previous round — each round produces progressively harder, more informative negatives.

This is fully configurable via YAML — just add `mining_rounds` to the data section.

In [ ]:
## Summary

| Model | Params | Strategy | nDCG@10 |
|-------|--------|----------|---------|
| BGE-base | 110M | No fine-tuning | (baseline) |
| MiniLM | 22M | No fine-tuning | (lower) |
| MiniLM | 22M | Random negatives | (improved) |
| MiniLM | 22M | Hard negatives | (improved) |
| MiniLM | 22M | Mixed negatives | (better) |
| MiniLM | 22M | Mixed + 2 mining rounds | (best) |

**Key takeaways:**
- A 5x smaller model, fine-tuned on domain data with LoRA, can approach the performance of a much larger general-purpose model.
- Mixed negatives (random + hard) provide the most balanced training signal.
- Iterative mining rounds progressively improve negative quality — each round re-mines using the fine-tuned model from the previous round, with LR automatically halved to avoid overshooting.

## Summary

| Model | Params | Fine-tuned | nDCG@10 |
|-------|--------|------------|--------|
| BGE-base | 110M | No | (baseline) |
| MiniLM | 22M | No | (lower) |
| MiniLM | 22M | Random negs | (improved) |
| MiniLM | 22M | Hard negs | (improved) |
| MiniLM | 22M | Mixed negs | (best) |

**Key takeaway:** A 5x smaller model, fine-tuned on domain data with LoRA, can approach the performance of a much larger general-purpose model. Mixed negatives (random + hard) provide the most balanced training signal.

In [ ]:
# Compare: does 2-round mining beat single-round?
print(f"{'Strategy':<25} {'nDCG@10':>10} {'MRR@10':>10} {'Recall@10':>10}")
print("-" * 57)
for name, metrics in [
    ("MiniLM baseline", minilm_baseline.metrics),
    ("+ mixed (1 round)", result_mixed_eval.metrics),
    ("+ mixed (2 rounds)", result_2rounds.finetuned.metrics),
    ("BGE-base (110M)", bge_baseline.metrics),
]:
    print(f"{name:<25} {metrics['ndcg@10']:>10.4f} {metrics['mrr@10']:>10.4f} {metrics['recall@10']:>10.4f}")